- _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_itemchild
- _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other
- _exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier
- _exponent._bronze_allscripts_tw_works_vw.dbo_cpt4_modifier_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_vendor_item

### Need
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_item
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_source_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_idx_user
- _exponent._bronze_allscripts_tw_works_vw.dbo_source_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_activity_type_de

### Maybe Need
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_item

In [0]:

source = 'allscripts_tw'

### Charge Based Procedures

In [0]:
WITH charge_procedures AS (
  SELECT
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.id AS STRING)                                  AS source_record_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.encounterid AS STRING)                         AS source_encounter_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.visitid AS STRING)                             AS source_visit_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingproviderid AS STRING)                   AS source_provider_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.otherproviderid AS STRING)                     AS source_performing_provider_id,
    DATE(COALESCE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm,
                  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime,
                  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime))                              AS procedure_date,
    COALESCE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm,
             _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime,
             _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime)                                    AS procedure_datetime,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.unitstobillfor AS DOUBLE)                      AS quantity,
    NULLIF(
      REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''),
      ''
    )                                                                                                       AS procedure_source_value,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.entryname AS STRING)                   AS procedure_source_name
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_charge.chargecodede =
       _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.id
  WHERE 1 = 1
    -- keep non-visit "procedure" charges
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.islevelofservicechargeflag = 'N'
    -- keep active-ish billing statuses (drop canceled/removed)
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingstatus NOT IN ('C','R')
    -- require CPT present on the charge code dictionary row
    AND NULLIF(
          REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''),
          ''
        ) IS NOT NULL
),

order_item_result_procedures AS (
  SELECT
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.id AS STRING)                   AS source_record_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.encounterid AS STRING)          AS source_encounter_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.visitid AS STRING)              AS source_visit_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.orderingproviderid AS STRING)          AS source_provider_id,
    DATE(_exponent._bronze_allscripts_tw_works_vw.dbo_item_result.performeddttm)                            AS procedure_date,
    _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.performeddttm                                  AS procedure_datetime,
    CAST(1 AS DOUBLE)                                                                                       AS quantity,
    COALESCE(
      NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.cpt4code,   '[\\s\\u00A0]+', ''), ''),
      NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
    )                                                                                                       AS procedure_source_value,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.entryname AS STRING)             AS procedure_source_name
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.orderactivityid =
       _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.currentorderactivityid
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.orderitemext =
       _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.ordernumberext
   AND _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.patientid =
       _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.patientid
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.id =
       _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.qoclassificationde
  WHERE 1 = 1
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.activitytype = 'Order'
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
    AND TRIM(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.itemtype) = 'OT'
    AND COALESCE(
          TRIM(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.ordertype),
          TRIM(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.ordertype)
        ) <> 'L'
    AND COALESCE(
          NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.cpt4code,   '[\\s\\u00A0]+', ''), ''),
          NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
        ) IS NOT NULL
),

all_procedure_candidates AS (
  SELECT * FROM order_item_result_procedures
  UNION ALL
  SELECT * FROM charge_procedures
),

procedure_occurrence_shaped AS (
  SELECT
    /* you can replace this with your standard silver surrogate key logic */
    CAST(NULL AS BIGINT)                                                                                   AS procedure_occurrence_id,

    /* TODO: join to your person crosswalk to populate person_id */
    CAST(NULL AS BIGINT)                                                                                   AS person_id,

    /* TODO: map procedure_source_value (CPT/HCPCS) -> standard OMOP concept_id */
    CAST(0 AS INT)                                                                                          AS procedure_concept_id,

    all_procedure_candidates.procedure_date                                                                  AS procedure_date,
    all_procedure_candidates.procedure_datetime                                                              AS procedure_datetime,

    /* common choices: EHR order (38000275), EHR billing record (often local). set per-source if you prefer */
    CAST(38000275 AS INT)                                                                                    AS procedure_type_concept_id,

    CAST(NULL AS INT)                                                                                        AS modifier_concept_id,
    CAST(all_procedure_candidates.quantity AS DOUBLE)                                                        AS quantity,

    /* TODO: join to provider crosswalk if you want provider_id */
    CAST(NULL AS BIGINT)                                                                                     AS provider_id,

    /* TODO: join to visit_occurrence via visit/encounter crosswalk */
    CAST(NULL AS BIGINT)                                                                                     AS visit_occurrence_id,
    CAST(NULL AS BIGINT)                                                                                     AS visit_detail_id,

    CAST(all_procedure_candidates.procedure_source_value AS STRING)                                          AS procedure_source_value,

    /* TODO: map CPT/HCPCS source concept if you maintain source_concept_id */
    CAST(0 AS INT)                                                                                           AS procedure_source_concept_id,

    CAST(NULL AS STRING)                                                                                     AS modifier_source_value,

    /* lineage fields you may want to keep in silver */
    CAST(all_procedure_candidates.source_record_id AS STRING)                                                AS source_record_id,
    CAST(all_procedure_candidates.source_visit_id AS STRING)                                                 AS source_visit_id,
    CAST(all_procedure_candidates.source_encounter_id AS STRING)                                             AS source_encounter_id,
    CAST(all_procedure_candidates.source_provider_id AS STRING)                                              AS source_provider_id,
    CAST(all_procedure_candidates.source_performing_provider_id AS STRING)                                   AS source_performing_provider_id,
    CAST(all_procedure_candidates.procedure_source_name AS STRING)                                           AS procedure_source_name
  FROM all_procedure_candidates
)

SELECT
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_record_id,
  source_visit_id,
  source_encounter_id,
  source_provider_id,
  source_performing_provider_id,
  procedure_source_name
FROM procedure_occurrence_shaped;


In [0]:
%sql
-- =========================================================
-- STEP 1: PICK ONE CHARGE ROW (START HERE)
-- Notes: Comments indicate intended OMOP.PROCEDURE_OCCURRENCE target fields
-- =========================================================
SELECT
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.id AS STRING)                AS id,                 -- source identifier (use as procedure_source_value or for source_record_id lineage; NOT an OMOP field by itself)
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.visitid AS STRING)           AS visitid,            -- maps -> OMOP.PROCEDURE_OCCURRENCE.visit_occurrence_id (via visitid -> your visit_occurrence PK logic)
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.encounterid AS STRING)       AS encounterid,        -- often used to help derive visit_occurrence_id / visit_detail_id (implementation-specific; not a direct OMOP procedure field)
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.chargecodede AS STRING)      AS chargecodede,       -- maps -> OMOP.PROCEDURE_OCCURRENCE.procedure_source_value (and/or used to look up procedure_concept_id)
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.islevelofservicechargeflag        AS islevelofservicechargeflag, -- filter logic: 'N' include for procedures; 'Y' typically exclude (E/M LOS)
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingstatus                     AS billingstatus,      -- ETL filter / data quality (not an OMOP procedure field)
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm                          AS postdttm,           -- candidate for maps -> procedure_date / procedure_datetime (fallback precedence)
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime                         AS starttime,          -- candidate for maps -> procedure_date / procedure_datetime (fallback precedence)
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime                           AS endtime,            -- candidate for maps -> procedure_date / procedure_datetime (fallback precedence)

  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.unitstobillfor AS DOUBLE)    AS unitstobillfor,     -- maps -> OMOP.PROCEDURE_OCCURRENCE.quantity (when units reflect count of procedure units)

  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingproviderid AS STRING) AS billingproviderid,  -- maps -> OMOP.PROCEDURE_OCCURRENCE.provider_id (if you choose billing provider as the procedure provider)
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.otherproviderid AS STRING)   AS otherproviderid,    -- maps -> OMOP.PROCEDURE_OCCURRENCE.provider_id (alternative: performing provider, if this is the performer in your data)

  DATE(
    COALESCE(
      _exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm,
      _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime,
      _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime
    )
  ) AS procedure_date,                                                                 -- maps -> OMOP.PROCEDURE_OCCURRENCE.procedure_date

  COALESCE(
    _exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm,
    _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime,
    _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime
  ) AS procedure_datetime                                                              -- maps -> OMOP.PROCEDURE_OCCURRENCE.procedure_datetime
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge
WHERE 1 = 1
  -- Recommended for PROCEDURE_OCCURRENCE extraction (exclude E/M Level-of-Service charges)
  AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.islevelofservicechargeflag = 'N'

  -- Common data quality / exclusion filters (optional)
  -- AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.id = <charge_id>
  -- AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingstatus NOT IN ('C','R');


In [0]:
%sql
-- =========================================================
-- STEP 2: LOOK UP THE CHARGE CODE DICTIONARY ROW
-- Replace <chargecodede_id> with the chargecodede value from Step 1
-- =========================================================
SELECT
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.id AS STRING)         AS id,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.entrycode AS STRING)  AS entrycode,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.entryname AS STRING)  AS entryname,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.cpt4code AS STRING)   AS cpt4code_raw,
  NULLIF(
    REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''),
    ''
  )                                                                                      AS cpt4code_norm,
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.islevelofserviceflag       AS islevelofserviceflag,
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.isinactiveflag             AS isinactiveflag,
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.iscurrentflag              AS iscurrentflag
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de
WHERE 1 = 1



In [0]:
%sql
-- =========================================================
-- STEP 3: PULL ANY MODIFIERS ON THAT CHARGE (OPTIONAL)
-- Replace <charge_id> with the same charge.id from Step 1
-- =========================================================
SELECT
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier.chargeid AS STRING)         AS chargeid,
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier.modifiernumber                   AS modifiernumber,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier.billingchargemodifierde AS STRING) AS billingchargemodifierde
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier
WHERE 1 = 1
  AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier.chargeid = <charge_id>
ORDER BY _exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier.modifiernumber;


In [0]:
%sql
-- =========================================================
-- STEP 4: PULL ANY CHARGE EDIT STATUSES (OPTIONAL)
-- Replace <charge_id> with the same charge.id from Step 1
-- =========================================================
SELECT
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_status.chargeid AS STRING)       AS chargeid,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_status.chargestatusde AS STRING) AS chargestatusde,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_status.conflictingchargeid AS STRING) AS conflictingchargeid
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge_status
WHERE 1 = 1
  AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge_status.chargeid = <charge_id>;

In [0]:
%sql
-- =========================================================
-- STEP 5: DECODE CHARGE EDIT STATUSES (OPTIONAL)
-- Replace <chargestatusde_id> with a chargestatusde from Step 4
-- =========================================================
SELECT
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_status_de.id AS STRING)         AS id,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_status_de.entrycode AS STRING)  AS entrycode,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_status_de.entryname AS STRING)  AS entryname,
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge_status_de.severity                  AS severity
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge_status_de
WHERE 1 = 1
  AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge_status_de.id = <chargestatusde_id>;

In [0]:
%sql
-- =========================================================
-- STEP 6: PULL ANY DIAGNOSES LINKED TO THE CHARGE (OPTIONAL)
-- Replace <charge_id> with the same charge.id from Step 1
-- =========================================================
SELECT
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.chargeid AS STRING)    AS chargeid,
  _exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.diagnosisnumber            AS diagnosisnumber,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.billingdiagnosistype AS STRING) AS billingdiagnosistype,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.icd10diagnosiscode AS STRING)   AS icd10diagnosiscode,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.icd9diagnosiscode AS STRING)    AS icd9diagnosiscode,
  CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.problemde AS STRING)            AS problemde
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis
WHERE 1 = 1
  AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.chargeid = <charge_id>
ORDER BY _exponent._bronze_allscripts_tw_works_vw.dbo_charge_diagnosis.diagnosisnumber;

In [0]:
%sql 
DESCRIBE TABLE _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other;



In [0]:
%sql
SELECT DISTINCT
  -- procedure_occurrence_id,

  CAST(dbo_encounter.patientid AS BIGINT) AS person_id,
  COALESCE(standard_concept.concept_id, 0) AS procedure_concept_id,

  CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS DATE)      AS procedure_date,
  CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS TIMESTAMP) AS procedure_datetime,

  CAST(44814649 AS INT) AS procedure_type_concept_id,
  CAST(0 AS INT) AS modifier_concept_id,
  CAST(COALESCE(dbo_charge.unitstobillfor, 1) AS DOUBLE) AS quantity,

  CAST(COALESCE(dbo_charge.otherproviderid, dbo_charge.billingproviderid) AS BIGINT) AS provider_id,
  CAST(dbo_charge.visitid AS BIGINT) AS visit_occurrence_id,
  CAST(NULL AS BIGINT) AS visit_detail_id,

  NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') AS procedure_source_value,
  COALESCE(concept.concept_id, 0) AS procedure_source_concept_id,

  primary_modifier.modifier_source_value AS modifier_source_value,
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de
  ON dbo_charge_code_de.id = dbo_charge.chargecodede

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other
  ON dbo_encounter_other.EncounterId = dbo_charge.encounterid

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  ON dbo_encounter.id = dbo_encounter_other.EncounterId

LEFT JOIN (
  SELECT
    cm.ChargeID,
    NULLIF(REGEXP_REPLACE(CAST(cmd.entrycode AS STRING), '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
    ROW_NUMBER() OVER (PARTITION BY cm.ChargeID ORDER BY cm.ModifierNumber ASC) AS rn
  FROM _exponent._bronze_allscripts_tw_works.dbo_charge_modifier cm
  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_cpt4_modifier_de cmd
    ON cmd.id = cm.BillingChargeModifierDE
) primary_modifier
  ON primary_modifier.ChargeID = dbo_charge.id
 AND primary_modifier.rn = 1

LEFT JOIN _exponent.omop.concept
  ON concept.concept_code = NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '')
 AND concept.vocabulary_id IN ('CPT4','HCPCS')
 AND concept.domain_id = 'Procedure'
 AND concept.invalid_reason IS NULL

LEFT JOIN _exponent.omop.concept_relationship
  ON concept_relationship.concept_id_1 = concept.concept_id
 AND concept_relationship.relationship_id = 'Maps to'

LEFT JOIN _exponent.omop.concept standard_concept
  ON standard_concept.concept_id = concept_relationship.concept_id_2
 AND standard_concept.standard_concept = 'S'
 AND standard_concept.invalid_reason IS NULL
 AND standard_concept.domain_id = 'Procedure'

WHERE 1 = 1
  AND dbo_charge.islevelofservicechargeflag = 'N'
  AND dbo_charge_code_de.IsLevelOfServiceFlag = 'N'
  AND NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') IS NOT NULL
  -- AND dbo_charge.etl_load_ts BETWEEN  CURRENT_DATE() - INTERVAL 14 DAY AND CURRENT_DATE();


### Activity Based Occurrences


### To Do:
 - Join to person mapping table on patientid
 - Join to provider mapping table on orderingproviderid
 - Join to visit_occurrence and visit_detail table on visitid

In [0]:
%sql
SELECT DISTINCT
  -- procedure_occurrence_id,

  -- CAST(dbo_encounter.patientid AS BIGINT) AS person_id,
  COALESCE(source_to_person.person_id, 0) AS person_id,
  COALESCE(standard_concept.concept_id, 0) AS procedure_concept_id,

  CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS DATE)      AS procedure_date,
  CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS TIMESTAMP) AS procedure_datetime,

  CAST(32817 AS INT) AS procedure_type_concept_id, -- hardcoding "EHR order"
  CAST(0 AS INT) AS modifier_concept_id,
  CAST(1 AS DOUBLE) AS quantity,

  CAST(dbo_order_activity.orderingproviderid AS BIGINT) AS provider_id,

  CAST(dbo_encounter.visitid AS BIGINT) AS visit_occurrence_id,
  CAST(dbo_encounter.visitid AS BIGINT) AS visit_detail_id,

  COALESCE(
    NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
    NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
  ) AS procedure_source_value,

  COALESCE(concept.concept_id, 0) AS procedure_source_concept_id,

  NULLIF(REGEXP_REPLACE(dbo_qo_mod_de.entrycode, '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
  ON dbo_order_activity.orderactivityheaderid = dbo_order_activity_header.id

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  ON dbo_encounter.id = dbo_order_activity_header.encounterid

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
  ON dbo_item_result.orderitemext = dbo_order_activity.ordernumberext
 AND dbo_item_result.patientid   = dbo_encounter.patientid

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de
  ON dbo_qo_classification_de.id = dbo_item_result.qoclassificationde

LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_mod_de
  ON dbo_qo_mod_de.id = COALESCE(dbo_item_result.qomod1de, dbo_item_result.qomod2de, dbo_item_result.qomod3de)

LEFT OUTER JOIN _exponent.omop.concept
  ON concept.concept_code = COALESCE(
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
      )
 AND concept.vocabulary_id IN ('CPT4','HCPCS')
 AND concept.domain_id = 'Procedure'
 AND concept.invalid_reason IS NULL

LEFT JOIN _exponent.omop.concept_relationship
  ON concept_relationship.concept_id_1 = concept.concept_id
 AND concept_relationship.relationship_id = 'Maps to'

LEFT JOIN _exponent.omop.concept standard_concept
  ON standard_concept.concept_id = concept_relationship.concept_id_2
 AND standard_concept.standard_concept = 'S'
 AND standard_concept.invalid_reason IS NULL
 AND standard_concept.domain_id = 'Procedure'
LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT('allscripts_tw | ', dbo_encounter.patientid)
LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT('allscripts_tw | ', dbo_order_activity.orderingproviderid)

WHERE 1 = 1
  AND dbo_order_activity_header.activitytype = 'Order'
  AND dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
  AND dbo_qo_classification_de.itemtype = 'OT'
  AND dbo_qo_classification_de.ordertype <> 'L'
  AND COALESCE(
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
      ) IS NOT NULL;
